# QAD recovery curves (vLLM evals)

Parses `qad/eval_results_vllm/` and plots benchmark accuracy vs QAD training step
for each quantization method, with the full-precision BF16 baseline as a horizontal line.

The grid has **one row per model size** (Qwen3-0.6B / 1.7B / 4B) and **one column per
benchmark**, plus a final **Average** column. The average covers exactly the benchmarks
left uncommented in `TASKS` (so commenting out e.g. AIME-2025 removes it from both the
panels and the mean), and is only plotted at steps where every one of them has a value.
Each model's own BF16 baseline is drawn per cell; methods share a color across the whole
grid (single legend on top).

In [5]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt

VLLM_DIR = Path("../qad/eval_results_vllm")

# Which training recipe to plot. Checkpoint tags are <RUN>-<model>-<method>-<hash>,
# so this switches between the two sets of runs. Uncomment exactly one.
RUN = "qad3x"    # 30M tokens, constant 3e-6 LR after 100-step warmup
# RUN = "qad"    # 10M tokens, cosine decay

# task -> (display title, preferred metric key). Metric keys vary by task filter:
#   - math500          : symbolic math_verify,none
#   - mmlu_flan_cot_*   : CoT generative MMLU; a GROUP whose top-level row is empty —
#                         the metric lives on 57 subject leaves (exact_match,flexible-extract),
#                         so get_metric() sample-weight-aggregates the leaves.
#   - mmlu_pro         : CoT; aggregates at the group level as exact_match,custom-extract.
# Comment a row out to drop it from the panels AND from the mean-recovery panel.
TASKS = {
    "gsm8k":                  ("GSM8K",      "exact_match,flexible-extract"),
    "minerva_math500":        ("MATH-500",   "math_verify,none"),
    # "aime25":                 ("AIME-2025",  "exact_match,none"),
    "mmlu_flan_cot_zeroshot": ("MMLU (CoT)", "exact_match,flexible-extract"),
    "mmlu_pro":               ("MMLU-Pro",   "exact_match,custom-extract"),
}

# quantizer -> (legend label, color). Plot order follows this dict.
# Comment a row out to hide that method; any method found on disk but missing here is
# reported by the warning below and simply not plotted.
#
# A key may be either "<method>" or "<method>-<hash>". The hash-qualified form wins
# when both are present, and is how you separate several runs of the SAME method that
# differ only in hyperparameters (the hash is over the quantizer params). Without it
# every arm of a sweep collapses onto one label/color — see the collision warning.
METHODS = {
    # Baselines
    "nvfp4":    ("NVFP4",  "tab:green"),
    "nvfp4a16": ("NVFP4A16", "tab:blue"),
    "lloyd3bit": ("W3A16 LLOYD SIGNED", "tab:purple"),

    # GSQ
    # "gsqlloyd3bit-34a2e5a5": ("GSQ LLOYD W3A16, logit_lr=3e-5", "tab:red"),
    # "gsqlloyd3bit-113e0057": ("GSQ LLOYD W3A16, logit_lr=1e-4", "tab:orange"),
    # "gsqlloyd3bit-28a6f8d3": ("GSQ LLOYD W3A16, logit_lr=3e-4", "tab:brown"),
    # "gsqlloyd3bit-2c7917b8": ("GSQ LLOYD W3A16, logit_lr=1e-5", "tab:cyan"),
    # "gsqlloyd3bit-c96c7cae": ("GSQ LLOYD W3A16, logit_lr=3e-6", "tab:olive"),
    # "gsqlloyd3bit-bcb432c4": ("GSQ LLOYD W3A16, logit_lr=1e-5, capped", "tab:pink"),
    # "gsqlloyd3bit-0a6afc4d": ("GSQ LLOYD W3A16, logit_lr=3e-6, capped", "tab:gray"),

    # Extra
    # "ste4bit":  ("W4A4 INT4 GS128",     "tab:orange"),
    # "ste3bit":  ("STE 3-bit",     "tab:red"),
    # "ste2bit":  ("STE 2-bit",     "tab:brown"),
    # "quest4bit":("QuEST 4-bit",   "tab:purple"),
    # "quest3bit":("QuEST 3-bit",   "tab:brown"),
    # "quest2bit":("QuEST 2-bit",   "tab:pink"),
    # "fp8":      ("FP8",           "tab:gray"),
}


def _leaf_metric(m, key):
    if key in m:
        return m[key]
    for k, v in m.items():
        if k.startswith(("exact_match,", "math_verify,")) and not k.endswith("_stderr"):
            return v
    return None


def get_metric(doc, task, key):
    results = doc.get("results", {})
    r = results.get(task, {})
    # 1) direct metric on the task / group aggregate row
    v = _leaf_metric(r, key)
    if v is not None:
        return v
    # 2) group with an empty top-level row: sample-weighted mean over subject leaves
    #    (keys like "<task>_abstract_algebra"; excludes "<task>::category" mid-groups).
    num = den = 0.0
    for t, m in results.items():
        if t == task or not t.startswith(task + "_"):
            continue
        lv = _leaf_metric(m, key)
        if lv is None:
            continue
        w = m.get("sample_len") or 1
        num += lv * w
        den += w
    return num / den if den else None


def load_tag(tag_dir):
    """Return {task: {step: accuracy_pct}} for one checkpoint-tag directory."""
    out = {}
    for jf in sorted(tag_dir.glob("step_*.json")):
        m = re.search(r"step_(\d+)", jf.name)
        if not m:
            continue
        step = int(m.group(1))
        doc = json.loads(jf.read_text())
        for task, (_title, key) in TASKS.items():
            v = get_metric(doc, task, key)
            if v is not None:
                out.setdefault(task, {})[step] = float(v) * 100.0
    return out


# Tag conventions (from eval_vllm.py):
#   method   : qad-<Model>-<quantizer>-<8hexhash>     e.g. qad-Qwen-Qwen3-4B-nvfp4-99914b93
#   baseline : <Model>-unquantized                    e.g. Qwen-Qwen3-1.7B-unquantized
# <Model> is e.g. Qwen-Qwen3-0.6B / Qwen-Qwen3-1.7B / Qwen-Qwen3-4B.
# The hash is captured too, so METHODS can key on "<method>-<hash>" (see above).
_METHOD_RE = re.compile(rf"^{RUN}-(Qwen-Qwen3-[^-]+)-([A-Za-z0-9.]+)-([0-9a-f]{{8}})$")
_BASELINE_RE = re.compile(r"^(Qwen-Qwen3-[^-]+)-unquantized$")

models = {}      # model -> {key -> {task -> {step -> acc_pct}}}   key: method or method-hash
baselines = {}   # model -> {task -> acc_pct}  (BF16, unquantized)
seen_hashes = {}  # method -> {hash} — to detect arms sharing one un-qualified key

for tag_dir in sorted(VLLM_DIR.glob("*")):
    if not tag_dir.is_dir():
        continue
    data = load_tag(tag_dir)
    if not data:
        continue
    mb = _BASELINE_RE.match(tag_dir.name)
    mm = _METHOD_RE.match(tag_dir.name)
    if mb:
        model = mb.group(1)
        for task, steps in data.items():
            if steps:
                baselines.setdefault(model, {})[task] = steps[min(steps)]  # single step-0 entry
    elif mm:
        model, method, qhash = mm.group(1), mm.group(2), mm.group(3)
        # hash-qualified key wins when listed; otherwise all hashes share one key
        key = f"{method}-{qhash}" if f"{method}-{qhash}" in METHODS else method
        seen_hashes.setdefault(method, set()).add(qhash)
        models.setdefault(model, {})[key] = data


def _size(model):  # sort key: 0.6 < 1.7 < 4
    m = re.search(r"Qwen3-([\d.]+)B", model)
    return float(m.group(1)) if m else 1e9


MODEL_ORDER = sorted(models, key=_size)

# Methods to plot: METHODS order, restricted to those actually present on disk.
found = {q for m in models.values() for q in m}
PLOT_METHODS = [q for q in METHODS if q in found]
unlisted = sorted(found - set(METHODS))
if unlisted:
    # suggest the hash-qualified name so it can be pasted straight into METHODS
    sugg = [f"{q}-{sorted(seen_hashes.get(q, {'?'}))[0]}" if len(seen_hashes.get(q, ())) > 1
            else q for q in unlisted]
    print(f"WARNING: results found for method(s) missing from METHODS — NOT plotted: "
          f"{unlisted}. Add them to METHODS to include them (e.g. {sugg}).")
missing = [q for q in METHODS if q not in found]
if missing:
    print(f"note: METHODS entries with no results on disk: {missing}")
# Several hyperparameter arms silently overwriting each other is the failure this
# guards: they'd share one label, one color, and only the last one read would survive.
for method, hashes in sorted(seen_hashes.items()):
    if len(hashes) > 1 and method in found:
        print(f"WARNING: {method!r} has {len(hashes)} hashes on disk {sorted(hashes)} but is "
              f"keyed WITHOUT a hash — only one arm is plotted. Replace the {method!r} entry "
              f"in METHODS with one '{method}-<hash>' entry per arm.")

print(f"run     : {RUN}")
print("models  :", {m.replace("Qwen-", ""): sorted(models[m]) for m in MODEL_ORDER})
print("plotting:", PLOT_METHODS)
print("baselines:", {m.replace("Qwen-", ""): {t: round(v, 1) for t, v in b.items()}
                     for m, b in baselines.items()})

run     : qad3x
models  : {'Qwen3-0.6B': ['gsqlloyd3bit', 'lloyd3bit', 'nvfp4', 'nvfp4a16', 'ste3bit', 'ste4bit'], 'Qwen3-1.7B': ['lloyd3bit', 'nvfp4', 'nvfp4a16', 'ste3bit', 'ste4bit'], 'Qwen3-4B': ['lloyd3bit', 'nvfp4', 'nvfp4a16', 'ste3bit', 'ste4bit']}
plotting: ['nvfp4', 'nvfp4a16', 'lloyd3bit']
baselines: {'Qwen3-0.6B': {'gsm8k': 65.4, 'minerva_math500': 60.2, 'mmlu_flan_cot_zeroshot': 45.5, 'mmlu_pro': 30.2}, 'Qwen3-1.7B': {'gsm8k': 79.2, 'minerva_math500': 76.4, 'mmlu_flan_cot_zeroshot': 58.7, 'mmlu_pro': 50.1}, 'Qwen3-4B': {'gsm8k': 89.2, 'minerva_math500': 82.4, 'mmlu_flan_cot_zeroshot': 74.9, 'mmlu_pro': 61.2}}


In [ ]:
# Clear message instead of an opaque "Number of rows must be a positive integer"
# from plt.subplots() when the selected RUN has no eval results yet.
assert MODEL_ORDER, (f"No eval results for RUN={RUN!r} yet "
                     "- evals may still be queued; flip the RUN toggle above.")

tasks = list(TASKS)                 # only the benchmarks left uncommented in TASKS
AVG = "__avg__"                     # synthetic panel: mean RECOVERY over those benchmarks
panels = tasks + [AVG]
MIN_STEP = 100

def avg_recovery(curves, bl):
    """Mean *recovery* over the plotted benchmarks: mean_t(acc_t / baseline_t) * 100.

    Averaging recovery ratios rather than raw accuracies keeps the benchmarks on a
    comparable footing — otherwise a high-scoring benchmark (GSM8K) dominates a
    low-scoring one (MMLU-Pro) and the mean mostly tracks the easy tasks. By
    construction the BF16 baseline sits at exactly 100%.

    Only steps where EVERY plotted benchmark has a value are averaged, so a
    partially-failed sweep can't make the curve jump. Returns ({step: pct}, skipped).
    """
    usable = [t for t in tasks if t in curves and bl.get(t)]   # needs a non-zero baseline
    if len(usable) < len(tasks):
        return {}, sorted({s for t in usable for s in curves[t]})
    complete = set.intersection(*(set(curves[t]) for t in tasks))
    skipped = set().union(*(set(curves[t]) for t in tasks)) - complete
    return ({s: 100.0 * sum(curves[t][s] / bl[t] for t in tasks) / len(tasks)
             for s in complete}, sorted(skipped))


nrows, ncols = len(MODEL_ORDER), len(panels)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 3.7 * nrows),
                         squeeze=False, sharex=True,
                        #  sharey="col",
                         sharey=None,
)
skipped_note = {}

for i, model in enumerate(MODEL_ORDER):
    bl_all = baselines.get(model, {})
    for j, panel in enumerate(panels):
        ax = axes[i][j]
        for method in PLOT_METHODS:            # METHODS order, only what's on disk
            label, color = METHODS[method]
            curves = models[model].get(method, {})
            if panel == AVG:
                curve, skip = avg_recovery(curves, bl_all)
                if skip:
                    skipped_note[(model, method)] = skip
            else:
                curve = curves.get(panel)
            if not curve:
                continue
            steps = sorted(curve)
            steps = [s for s in steps if s >= MIN_STEP]
            ax.plot(steps, [curve[s] for s in steps], marker="o", markersize=4,
                    linewidth=1.6, color=color, label=label)
        # baseline: raw accuracy per benchmark; exactly 100% recovery on the average
        bl = 100.0 if panel == AVG else bl_all.get(panel)
        if bl is not None:
            ax.axhline(bl, ls="--", color="0.4", linewidth=1.3)
            ax.annotate("BF16 100%" if panel == AVG else f"BF16 {bl:.1f}%",
                        xy=(0.98, bl), xycoords=("axes fraction", "data"),
                        ha="right", va="bottom", fontsize=7, color="0.4")
        if i == 0:
            ax.set_title(f"Mean recovery ({len(tasks)} benchmarks)" if panel == AVG
                         else TASKS[panel][0],
                         fontsize=12, fontweight="bold" if panel == AVG else "normal")
        if j == 0:
            ax.set_ylabel(model.replace("Qwen-", "") + "\naccuracy (%)", fontsize=10)
        if panel == AVG:
            ax.set_ylabel("recovery (% of BF16)", fontsize=9)
            ax.yaxis.set_label_position("right")
            # ax.set_ylim(70, None)
        if i == nrows - 1:
            ax.set_xlabel("QAD step")
        ax.grid(True, alpha=0.3)
        ax.set_xlim(MIN_STEP, None)
        

# one shared legend for the whole grid (labels/colors from METHODS)
handles = [plt.Line2D([], [], color=METHODS[q][1], marker="o", label=METHODS[q][0])
           for q in PLOT_METHODS]
handles.append(plt.Line2D([], [], color="0.4", ls="--", label="BF16 baseline"))
fig.legend(handles=handles, loc="upper center", ncol=len(handles), fontsize=10,
           bbox_to_anchor=(0.5, 1.005), frameon=False)
fig.suptitle("QAD accuracy recovery vs training step (vLLM eval)   |   mean recovery over: "
             + ", ".join(TASKS[t][0] for t in tasks), y=1.03, fontsize=13)
fig.tight_layout()
fig.savefig("qad_recovery_curves.png", dpi=150, bbox_inches="tight")
plt.show()

for (model, method), steps in sorted(skipped_note.items()):
    print(f"note: {model.replace('Qwen-', '')} {METHODS[method][0]} — steps {steps} "
          f"not averaged (missing one of {[TASKS[t][0] for t in tasks]})")